# S03 · 01 — Entrenar sin tracking (el dolor)

Este notebook **no usa MLflow**. Existe para que el problema se sienta antes de
ver la herramienta que lo resuelve.

## Qué vas a hacer

1. Entrenar tres modelos con hiperparámetros distintos.
2. Intentar responder cinco preguntas sobre esos tres entrenamientos usando solo
   lo que quedó en la pantalla.
3. Fallar en las cinco.

## Antes de empezar

Las particiones del caso guía tienen que estar materializadas. Desde la raíz del
repositorio:

```bash
make data     # equivale a: uv run taxi data
```

No hace falta ningún servidor: en este notebook no hay nada que registrar.

## 1. Los datos vienen del paquete, no de este notebook

`taxi.models.train` resuelve las particiones (`data/processed/2023-01.parquet`, …)
y `taxi.features.contract` define las features. Es una regla del curso, no una
comodidad: antes del rediseño este módulo tenía **dos** rutas de
preprocesamiento incompatibles para el mismo problema —una con `pickle` y
`DictVectorizer`, otra con `parquet` y `ColumnTransformer`— y los ejercicios
leían de una mientras los notebooks escribían en la otra.

El notebook explora y narra. La lógica vive en `src/taxi/`.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

from taxi import config
from taxi.features import contract as fc
from taxi.models import train

df_train = train.cargar_train()
df_valid = train.cargar_valid()

print("train:", df_train.shape, "particiones:", [p.etiqueta for p in config.PARTICIONES_TRAIN])
print("valid:", df_valid.shape, "particion:", config.PARTICION_VALID.etiqueta)
print("features:", fc.FEATURES)
print("target:", fc.TARGET_REGRESION)

> El holdout (`PARTICION_TEST`, 2023-05) **no se carga aquí**. Es el juez del gate
> de promoción de S06 y cada mirada le gasta capacidad de estimar
> generalización.

## 2. Un vistazo al target

No es EDA completo: es lo mínimo para poder interpretar un RMSE en minutos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(df_train[fc.TARGET_REGRESION], bins=60)
ejes[0].set_title("Duracion en entrenamiento (min)")
ejes[0].set_xlabel("minutos")

ejes[1].hist(df_train["trip_distance"], bins=60, range=(0, 15))
ejes[1].set_title("Distancia (millas, recortado a 15)")
ejes[1].set_xlabel("millas")
fig.tight_layout()
plt.show()

print(df_train[[fc.TARGET_REGRESION, "trip_distance"]].describe().round(2))

El rango está acotado entre 1 y 60 minutos porque el contrato de datos de S02
descarta lo de fuera: viajes de 0 minutos y de 8 horas son errores de captura,
no viajes. Ese filtro es parte del **contrato**, no de este notebook.

## 3. La vara de medir: el baseline trivial

Antes de cualquier modelo, el número contra el que se interpreta todo lo demás:
predecir siempre la duración media. Si un modelo no le gana, no está aprendiendo
nada del dato.

In [ ]:
baseline = train.pipeline_media()
train.ajustar(baseline, df_train)

y_valid = df_valid[fc.TARGET_REGRESION].to_numpy(dtype=float)
rmse_media = float(root_mean_squared_error(y_valid, baseline.predict(df_valid)))
print(f"RMSE del baseline (predecir la media): {rmse_media:.4f} min")

## 4. Tres entrenamientos, tres `print`

Así se ve el trabajo real antes de tener tracking: se cambia un hiperparámetro,
se corre, se mira el número, se cambia otro.

> **Nota de tiempo:** se usan 25 árboles para que la celda tarde alrededor de un
> minuto. El default del paquete son 100 (`train.PARAMS_RANDOM_FOREST`).
> Cronométrala en tu máquina: el número depende de tus núcleos.

In [ ]:
configuraciones = [
    {"max_depth": 5, "n_estimators": 25},
    {"max_depth": 10, "n_estimators": 25},
    {"max_depth": 20, "n_estimators": 25},
]

for params in configuraciones:
    pipeline = train.pipeline_random_forest(**params)
    train.ajustar(pipeline, df_train, df_valid)
    rmse = float(root_mean_squared_error(y_valid, pipeline.predict(df_valid)))
    # Un print. Sin fecha, sin datos, sin version de codigo, sin artefacto.
    print(f"{params} -> RMSE={rmse:.4f}")

## 5. Las cinco preguntas

Mirando **solo** la salida de la celda anterior, responde:

1. ¿Cuál de las tres configuraciones fue la mejor, y por cuánto?
2. ¿Con qué particiones de datos se entrenó la segunda?
3. ¿Con qué versión del código? ¿Había cambios sin commitear?
4. ¿Dónde está el modelo de la mejor configuración? ¿Se puede cargar hoy?
5. ¿Se puede repetir exactamente la tercera corrida dentro de tres meses?

La 1 se responde a duras penas. Las otras cuatro, no.

Y hay un agravante: **reinicia el kernel**. Los tres modelos desaparecen y solo
queda el texto de la salida, que además se pierde en cuanto alguien vuelva a
ejecutar la celda.

## 6. Lo que hace falta

| Pregunta | Qué hace falta registrar |
|---|---|
| ¿cuál fue mejor? | **métricas** comparables entre corridas |
| ¿con qué configuración? | **parámetros** |
| ¿con qué datos? | **tags** con las particiones y el hash del dato |
| ¿dónde está el modelo? | **artifacts**: el modelo serializado, con su `signature` |
| ¿se puede repetir? | la unión de todo lo anterior más la semilla y la versión del código |

Eso es el experiment tracking, y es lo que hace el siguiente notebook:
[`02-tracking-con-mlflow.ipynb`](02-tracking-con-mlflow.ipynb).

Un cierre honesto: nada de esto mejora el modelo. Un experimento bien registrado
con un modelo malo sigue siendo un modelo malo, pero es un modelo malo que se
puede comparar, reproducir y descartar con evidencia.